# LSTM ile meteorolojik sıcaklık tahmini
### Yüksek lisans vize sunumu · Jena Climate · TensorFlow/Keras

**Araştırma sorusu:** Son 24 saatin meteorolojik ölçümleriyle bir saat sonraki sıcaklık tahmin edilebilir mi? LSTM, son sıcaklığı tekrar eden basit baseline'dan daha iyi mi?

**Sunum akışı:** Problem → Veri → Ön işleme → Model → Eğitim → Sonuçlar → Eleştirel değerlendirme.

> Bu notebook sunum içindir; ayrıntılı kişisel öğrenme rehberi `learning/lstm_interactive_guide.html` dosyasındadır. **Önce** `python -m src.train --config config.yaml` komutunu çalıştırın. Sonuçlar üretilmediyse aşağıdaki hücreler bunu açıkça belirtir.

## 1. Problem ve motivasyon

Meteorolojik değişkenler zaman içinde birbirine bağlıdır. LSTM, önceki gözlemlerden bir **gizli durum** ve **hücre belleği** oluşturarak zaman örüntülerini öğrenmeyi hedefler.

**Tahmin görevi:** $X \in \mathbb{R}^{24 \times 5}$ geçmiş saatlik özellik → $\hat y \in \mathbb{R}$ bir saat sonrası hava sıcaklığı (°C).

## 2. Veri seti: Jena Climate

- Kaynak: Max Planck Institute for Biogeochemistry / TensorFlow veri dağıtımı.
- Orijinal ölçümler yaklaşık 10 dakika aralıklıdır; her 6. satır alınarak saatlik örnek kullanılır.
- Kullanılan özellikler: basınç, sıcaklık, bağıl nem, rüzgâr hızı, rüzgâr yönü.
- `T (degC)` hem geçmiş giriş özelliği hem gelecekteki hedef değişkendir.

**Neden bu veri?** Gerçek meteorolojik ölçümler, indirilebilir veri ve belirgin zamansal örüntüler.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import yaml
import matplotlib.pyplot as plt

# Notebook, repo kökünden veya notebooks/ klasöründen açılabilir.
ROOT = Path.cwd()
if not (ROOT / 'config.yaml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'config.yaml').exists(), 'Notebooku proje klasöründen açın.'
config = yaml.safe_load((ROOT / 'config.yaml').read_text(encoding='utf-8'))
config

## 3. Keşifsel veri analizi

Veri setinin zaman sırasını, temel istatistiklerini ve sıcaklığın zaman içinde nasıl değiştiğini inceliyoruz. Bu kısım eğitim ve test arasında **gelecek bilgisini öğrenmek** için kullanılmaz; yalnızca tanıtım/keşif içindir.

In [ ]:
csv_path = ROOT / 'data/raw' / config['data']['csv_name']
if not csv_path.exists():
    raise FileNotFoundError('CSV yok. Önce: python -m src.train --config config.yaml')
df = pd.read_csv(csv_path)
print('Ham veri boyutu:', df.shape)
display(df.head())
display(df[config['data']['features']].describe().round(2))
ax = df.iloc[::6].iloc[:24*14].plot(y='T (degC)', figsize=(12, 3), legend=False)
ax.set(title='Örnek: İlk iki haftanın saatlik sıcaklığı', xlabel='Saatlik örnek', ylabel='°C')
plt.show()

## 4. Ön işleme ve veri sızıntısını önleme

1. Geçersiz rüzgâr değeri `-9999` → eksik değer → ileri doldurma.
2. **Zamansal %70 / %20 / %10** eğitim, doğrulama, test bölmesi.
3. **StandardScaler yalnız eğitim** verisine fit edilir.
4. Kaydırmalı pencere: 24 saatlik giriş, izleyen 1 saatlik hedef.

**Kritik nokta:** Rastgele bölme yapılmaz. Doğrulama/test hedefleri ayrı tutulur; pencere girdisinin daha erken tarihlerden gelmesine izin verilir.

## 5. LSTM mimarisi ve matematik

\[
\begin{aligned}
f_t &= \sigma(W_f [h_{t-1},x_t]+b_f) \\
i_t &= \sigma(W_i [h_{t-1},x_t]+b_i) \\
\tilde c_t &= \tanh(W_c [h_{t-1},x_t]+b_c) \\
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde c_t \\
o_t &= \sigma(W_o [h_{t-1},x_t]+b_o) \\
h_t &= o_t \odot \tanh(c_t)
\end{aligned}
\]

**Mimari:** `Input(24,5) → LSTM(32) → Dropout(0.2) → Dense(1)`.

- **Unutma kapısı:** Eski bilginin tutulmasını belirler.
- **Giriş kapısı:** Yeni bilginin belleğe yazılmasını düzenler.
- **Çıkış kapısı:** Gizli durumun aktarılmasını belirler.
- **Adam + MSE:** Regresyon optimizasyonu.
- **EarlyStopping:** Doğrulama kaybı iyileşmezse eğitimi durdurur.

In [ ]:
import sys
sys.path.insert(0, str(ROOT))
from src.models import build_lstm
m = build_lstm(lookback=config['data']['lookback'],
               num_features=len(config['data']['features']),
               units=config['model']['units'],
               dropout=config['model']['dropout'],
               learning_rate=config['training']['learning_rate'])
m.summary()

## 6. Deney düzeni

Eğitim sırasında doğrulama kaybı takip edilir. En iyi model `.keras` olarak kaydedilir. Test bölmesi **yalnızca eğitimin sonunda** değerlendirilir.

**Naif baseline:** Bir sonraki sıcaklığın son gözlenen sıcaklığa eşit olduğunu kabul eder. LSTM performansı mutlaka bu basit referansla birlikte yorumlanmalıdır.

## 7. Eğitim grafiği ve gerçek tahminler

Aşağıdaki grafikler **gerçek eğitim çalıştırıldıktan sonra** `outputs/figures/` klasöründe oluşturulur; notebook bunları okur.

In [ ]:
from IPython.display import display, Image
for name in ['training_history.png', 'predictions.png']:
    path = ROOT / 'outputs/figures' / name
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print(f'Üretilmemiş sonuç: {name}. Önce eğitim çalıştırılmalı.')

## 8. Test sonuçları ve baseline karşılaştırması

- **MAE (°C):** Ortalama mutlak hata; doğrudan yorumlanabilir.
- **RMSE (°C):** Büyük hatalara daha duyarlı.
- **R²:** Hedef varyansına göre açıklama düzeyi.

> Aşağıdaki tablo bir gösterim değildir; yalnızca eğitim scriptinin kaydettiği gerçek değerlerden üretilir.

In [ ]:
metrics_path = ROOT / 'outputs/metrics.json'
if metrics_path.exists():
    scores = json.loads(metrics_path.read_text(encoding='utf-8'))
    display(pd.DataFrame(scores).T.round(4))
    if 'lstm' in scores and 'persistence_baseline' in scores:
        improvement = scores['persistence_baseline']['mae_c'] - scores['lstm']['mae_c']
        print(f'MAE farkı (baseline - LSTM): {improvement:+.3f} °C')
        print('Pozitif fark LSTM lehinedir; negatif fark baseline lehinedir.')
else:
    print('Henüz gerçek test sonucu yok. Önce python -m src.train komutunu çalıştırın.')

## 9. Eleştirel tartışma ve sınırlamalar

**LSTM daha iyi çıkarsa:** Modelin ek meteorolojik değişkenlerden ve zamansal örüntülerden yararlandığı hipoteziyle uyumludur; ancak tek veri bölmesi nedensellik kanıtlamaz.

**Baseline daha iyi çıkarsa:** Basit tahminin güçlü olduğu, mevcut LSTM ayarlarının bu problem için yeterli ek yarar sağlamadığı söylenmelidir. Bu da geçerli bir bilimsel sonuçtur.

**Sınırlamalar:** Tek istasyon; yalnız tek adımlı tahmin; rüzgâr yönü için döngüsel sin/cos kodlaması yok; 24 saatlik sabit pencere; tek seed/sabit tarih bölmesi; çoklu deney/güven aralığı yok.

**Gelecek çalışma:** Daha fazla pencere uzunluğu, mevsimsel naive baseline, çok adımlı tahmin ve zamansal çapraz doğrulama.

## 10. Sonuç ve kaynaklar

Bu çalışmada 24 saatlik 5 değişkenden bir saat sonrası sıcaklığın tahmini için bir LSTM hattı oluşturuldu. **Başarı iddiası ancak gerçek test metriği, grafik ve baseline karşılaştırması görüldükten sonra** kurulmalıdır.

**Kaynaklar**
1. Hochreiter, S. & Schmidhuber, J. (1997). *Long Short-Term Memory*. Neural Computation. https://doi.org/10.1162/neco.1997.9.8.1735
2. TensorFlow, *Time series forecasting tutorial*: https://www.tensorflow.org/tutorials/structured_data/time_series
3. Jena Climate veri kümesi: https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip

**Olası sözlü soru:** “Neden random split yapmadın?” → Gelecekten geçmişe bilgi sızmasını engellemek için kronolojik bölme kullandım.